# 🧠 ICT / SMC Concepts — Testing Notebook
**Author:** Next Level Brain  
**Purpose:** Test all ICT/SMC trading concepts independently from the live bot

---
## 📚 Concepts Covered
| # | Concept | Description |
|---|---------|-------------|
| 1 | MT5 Setup | Connect & Load Data |
| 2 | Market Structure (MSS/BOS/ChoCH) | Trend identification |
| 3 | Liquidity Sweep | Stop Hunt detection |
| 4 | Fair Value Gap (FVG) | Price imbalance zones |
| 5 | Dealing Range | Premium vs Discount |
| 6 | Order Block | Institutional footprint |
| 7 | OTE Levels | Fibonacci 62%-79% |
| 8 | Silver Bullet Windows | High-probability time zones |
| 9 | Liquidity Pool | Take Profit targeting |
| 10 | Full Confluence Engine | Combined signal generator |
| 11 | Walk-Forward Backtest | Historical performance |

> **How to run:** Click each cell and press `Shift+Enter` to execute one by one.

In [1]:
# ═══════════════════════════════════════════════════
# CELL 1 — IMPORTS & CONFIGURATION
# ═══════════════════════════════════════════════════
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv

load_dotenv()

# ─── User Config ─────────────────────────────────────
SYMBOL    = "BTCUSDm"          # Change to your symbol
TIMEFRAME = mt5.TIMEFRAME_M5   # M1, M5, M15, H1 etc.
BARS      = 500                # How many candles to load

print("✅ Imports loaded successfully")
print(f"   Symbol   : {SYMBOL}")
print(f"   Timeframe: {TIMEFRAME}")
print(f"   Bars     : {BARS}")

✅ Imports loaded successfully
   Symbol   : BTCUSDm
   Timeframe: 5
   Bars     : 500


In [2]:
# ═══════════════════════════════════════════════════
# CELL 2 — MT5 CONNECTION & DATA LOADER
# ═══════════════════════════════════════════════════
def connect_mt5():
    terminal_path = r"C:\Program Files\MetaTrader 5 EXNESS\terminal64.exe"
    if not mt5.initialize(path=terminal_path):
        raise RuntimeError(f"MT5 init failed: {mt5.last_error()}")
    login    = int(os.getenv("MT5_LOGIN", "0"))
    password = os.getenv("MT5_PASSWORD", "")
    server   = os.getenv("MT5_SERVER", "")
    if login and password and server:
        if not mt5.login(login, password=password, server=server):
            print(f"⚠️ Login failed: {mt5.last_error()}")
    acc = mt5.account_info()
    if acc:
        print(f"✅ Connected | Account: {acc.login} | Balance: ${acc.balance:.2f}")
    return True

def get_data(symbol=SYMBOL, tf=TIMEFRAME, bars=BARS):
    rates = mt5.copy_rates_from_pos(symbol, tf, 0, bars)
    if rates is None or len(rates) == 0:
        raise ValueError(f"No data returned for {symbol}")
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)
    print(f"📊 {len(df)} bars loaded | Latest close: {df['close'].iloc[-1]:.2f} @ {df.index[-1]}")
    return df

connect_mt5()
df = get_data()

✅ Connected | Account: 260150554 | Balance: $2064.09
📊 500 bars loaded | Latest close: 67336.27 @ 2026-03-07 23:15:00


---
## 📐 Concept 1 — Market Structure (MSS / BOS / ChoCH)
ICT uses **market structure** to determine the current trend direction.
- **BOS (Break of Structure)** = A swing high/low is broken, trend continues
- **ChoCH (Change of Character)** = Opposite structure broken, trend reversing
- **MSS (Market Structure Shift)** = First ChoCH after a liquidity sweep

In [3]:
# ═══════════════════════════════════════════════════
# CELL 3 — MARKET STRUCTURE SHIFT (MSS / BOS / ChoCH)
# ═══════════════════════════════════════════════════
def detect_market_structure(df, lookback=20):
    index  = len(df) - 1
    recent = df.iloc[max(0, index-lookback): index+1]
    swing_highs = recent['high'].rolling(3, center=True).max()
    swing_lows  = recent['low'].rolling(3, center=True).min()
    prev_high   = swing_highs.iloc[-2]
    prev_low    = swing_lows.iloc[-2]
    curr_high   = df['high'].iloc[-1]
    curr_low    = df['low'].iloc[-1]

    if   curr_high > prev_high and curr_low > prev_low: structure = "BULLISH_BOS"
    elif curr_low  < prev_low  and curr_high < prev_high: structure = "BEARISH_BOS"
    elif curr_low  < prev_low  and curr_high > prev_high: structure = "CHOCH"
    else: structure = "NEUTRAL"

    emoji = {"BULLISH_BOS": "🟢", "BEARISH_BOS": "🔴", "CHOCH": "🟡", "NEUTRAL": "⚪"}
    print(f"{emoji.get(structure,'⚪')} Market Structure : {structure}")
    print(f"   Prev High : {prev_high:.2f}  |  Curr High : {curr_high:.2f}")
    print(f"   Prev Low  : {prev_low:.2f}  |  Curr Low  : {curr_low:.2f}")
    return {'structure': structure, 'prev_high': prev_high, 'prev_low': prev_low}

mss = detect_market_structure(df)

⚪ Market Structure : NEUTRAL
   Prev High : 67375.64  |  Curr High : 67375.32
   Prev Low  : 67288.39  |  Curr Low  : 67308.49


---
## 💧 Concept 2 — Liquidity Sweep (Stop Hunt)
Banks push price **below swing lows** (to grab sell-stop orders) then reverse UP.  
Or push price **above swing highs** (buy-stops) then reverse DOWN.  
The sweep wick + close back inside = **high-probability reversal signal**.

In [4]:
# ═══════════════════════════════════════════════════
# CELL 4 — LIQUIDITY SWEEP DETECTION
# ═══════════════════════════════════════════════════
def detect_liquidity_sweep(df, lookback=20):
    index   = len(df) - 1
    recent  = df.iloc[max(0, index-lookback): index]
    swing_low  = recent['low'].min()
    swing_high = recent['high'].max()
    current = df.iloc[index]

    if current['low'] < swing_low and current['close'] > swing_low:
        result = {'detected': True, 'type': 'BELOW_LOW', 'swept_level': swing_low, 'strength': 0.8}
        print(f"💧 BULLISH Liquidity Sweep (Sell-Side Swept)")
        print(f"   Swing Low  : {swing_low:.2f}")
        print(f"   Candle Low : {current['low']:.2f}  →  Close: {current['close']:.2f}")
        print(f"   Signal     : ✅ BULLISH reversal likely")
    elif current['high'] > swing_high and current['close'] < swing_high:
        result = {'detected': True, 'type': 'ABOVE_HIGH', 'swept_level': swing_high, 'strength': 0.8}
        print(f"💧 BEARISH Liquidity Sweep (Buy-Side Swept)")
        print(f"   Swing High : {swing_high:.2f}")
        print(f"   Candle High: {current['high']:.2f}  →  Close: {current['close']:.2f}")
        print(f"   Signal     : ✅ BEARISH reversal likely")
    else:
        result = {'detected': False, 'type': None}
        print(f"💧 No Liquidity Sweep on current bar")
        print(f"   Price must break & close back inside swing high/low")
    return result

sweep = detect_liquidity_sweep(df)

💧 No Liquidity Sweep on current bar
   Price must break & close back inside swing high/low


---
## 📊 Concept 3 — Fair Value Gap (FVG / Imbalance)
FVG = 3-candle pattern where an **imbalance** (gap) exists between candle 1 and candle 3.  
Price tends to **retrace into the gap** before continuing.  
- **Bullish FVG**: Gap between C1.high and C3.low → price came up fast
- **Bearish FVG**: Gap between C1.low and C3.high → price came down fast

In [5]:
# ═══════════════════════════════════════════════════
# CELL 5 — FAIR VALUE GAP (FVG) + ACTIVE FVG SCANNER
# ═══════════════════════════════════════════════════

def _is_fvg_active(df, fvg_low, fvg_high, created_at_bar):
    """Check if price has NOT re-entered the FVG zone after creation."""
    bars_after = df.iloc[created_at_bar + 1:]
    if bars_after.empty:
        return True
    # FVG is filled if any subsequent bar overlaps the gap zone
    filled = ((bars_after['low'] <= fvg_high) & (bars_after['high'] >= fvg_low)).any()
    return not filled


def scan_active_fvgs(df, lookback=150, min_gap_pct=0.0005):
    """
    Scan historical bars and return ALL unfilled (active) FVGs.
    A FVG is considered ACTIVE if no candle after its creation
    has overlapped the gap zone (i.e., it has NOT been filled yet).
    """
    current_price = float(df['close'].iloc[-1])
    active_fvgs   = []
    start = max(3, len(df) - lookback)

    for i in range(start, len(df) - 1):
        b1, b2, b3 = df.iloc[i-2], df.iloc[i-1], df.iloc[i]
        min_size = b2['close'] * min_gap_pct

        # --- Bullish FVG ---
        if b1['high'] < b3['low']:
            gap = b3['low'] - b1['high']
            if gap > min_size and _is_fvg_active(df, b1['high'], b3['low'], i):
                dist = (current_price - b3['low']) / current_price * 100
                active_fvgs.append({
                    'type': 'BULLISH', 'low': b1['high'], 'high': b3['low'],
                    'gap_size': gap, 'strength': min(gap / (b2['close'] * 0.002), 1.0),
                    'bar_idx': i, 'bar_time': str(df.index[i]), 'distance_pct': dist
                })

        # --- Bearish FVG ---
        if b1['low'] > b3['high']:
            gap = b1['low'] - b3['high']
            if gap > min_size and _is_fvg_active(df, b3['high'], b1['low'], i):
                dist = (b3['high'] - current_price) / current_price * 100
                active_fvgs.append({
                    'type': 'BEARISH', 'low': b3['high'], 'high': b1['low'],
                    'gap_size': gap, 'strength': min(gap / (b2['close'] * 0.002), 1.0),
                    'bar_idx': i, 'bar_time': str(df.index[i]), 'distance_pct': dist
                })

    return active_fvgs


def detect_fair_value_gap(df):
    """Current bar FVG (last 3 bars only) — used by confluence engine."""
    index = len(df) - 1
    if index < 3:
        return {'detected': False}
    b1, b2, b3 = df.iloc[index-2], df.iloc[index-1], df.iloc[index]
    if b1['high'] < b3['low']:
        gap = b3['low'] - b1['high']
        if gap > b2['close'] * 0.0005:
            s = min(gap / (b2['close'] * 0.002), 1.0)
            return {'detected': True, 'type': 'BULLISH', 'high': b3['low'], 'low': b1['high'], 'gap_size': gap, 'strength': s}
    if b1['low'] > b3['high']:
        gap = b1['low'] - b3['high']
        if gap > b2['close'] * 0.0005:
            s = min(gap / (b2['close'] * 0.002), 1.0)
            return {'detected': True, 'type': 'BEARISH', 'high': b1['low'], 'low': b3['high'], 'gap_size': gap, 'strength': s}
    return {'detected': False}


# ── Run ──────────────────────────────────────────────
fvg           = detect_fair_value_gap(df)          # Current bar FVG
active_fvgs   = scan_active_fvgs(df, lookback=150) # All unfilled FVGs
current_price = float(df['close'].iloc[-1])

print('📊 Current Bar FVG  :', ('✅ ' + fvg['type']) if fvg['detected'] else '❌ None (last 3 bars)')
print()
print(f'📊 Active (Unfilled) FVGs in last 150 bars: {len(active_fvgs)} found')
print('─' * 58)

if active_fvgs:
    bullish_fvgs = [f for f in active_fvgs if f['type'] == 'BULLISH']
    bearish_fvgs = [f for f in active_fvgs if f['type'] == 'BEARISH']

    print(f'   🟢 Bullish FVGs (Below current price - BUY magnets): {len(bullish_fvgs)}')
    for f in sorted(bullish_fvgs, key=lambda x: abs(x['distance_pct']))[:5]:
        tag = '  ⬅ INSIDE!' if f['low'] <= current_price <= f['high'] else ''
        print(f"      {f['low']:.2f} – {f['high']:.2f}  | Gap: {f['gap_size']:.2f} | {f['distance_pct']:+.2f}% from price{tag}")

    print()
    print(f'   🔴 Bearish FVGs (Above current price - SELL magnets): {len(bearish_fvgs)}')
    for f in sorted(bearish_fvgs, key=lambda x: abs(x['distance_pct']))[:5]:
        tag = '  ⬅ INSIDE!' if f['low'] <= current_price <= f['high'] else ''
        print(f"      {f['low']:.2f} – {f['high']:.2f}  | Gap: {f['gap_size']:.2f} | {f['distance_pct']:+.2f}% from price{tag}")
else:
    print('   No active FVGs found in lookback window.')

print('─' * 58)
print(f'   Current Price : {current_price:.2f}')

# Auto-use nearest active FVG for OTE cell if current bar has none
if not fvg['detected'] and active_fvgs:
    nearest = min(active_fvgs, key=lambda x: abs(x['distance_pct']))
    fvg = {**nearest, 'detected': True}
    print(f"\n✅ Nearest active FVG used for OTE: {nearest['type']} @ {nearest['low']:.2f}–{nearest['high']:.2f}")


📊 Current Bar FVG  : ❌ None (last 3 bars)

📊 Active (Unfilled) FVGs in last 150 bars: 1 found
──────────────────────────────────────────────────────────
   🟢 Bullish FVGs (Below current price - BUY magnets): 0

   🔴 Bearish FVGs (Above current price - SELL magnets): 1
      67786.14 – 67851.40  | Gap: 65.26 | +0.67% from price
──────────────────────────────────────────────────────────
   Current Price : 67336.27

✅ Nearest active FVG used for OTE: BEARISH @ 67786.14–67851.40


---
## 📏 Concept 4 — Dealing Range (Premium vs Discount)
ICT divides the trading range into zones using the **50% equilibrium**:
- **Discount** (below 50%): Ideal zone to **BUY**
- **Premium** (above 50%): Ideal zone to **SELL**
- Buy in Discount + Sell in Premium = trade with institutional flow

In [6]:
# ═══════════════════════════════════════════════════
# CELL 6 — DEALING RANGE (PREMIUM / DISCOUNT)
# ═══════════════════════════════════════════════════
def analyze_dealing_range(df, lookback=20):
    index   = len(df) - 1
    recent  = df.iloc[max(0, index-lookback): index+1]
    rh      = float(recent['high'].max())
    rl      = float(recent['low'].min())
    current = float(df['close'].iloc[-1])
    rng     = rh - rl

    eq   = rl + rng * 0.50
    l25  = rl + rng * 0.25
    l75  = rl + rng * 0.75
    pct  = (current - rl) / rng * 100 if rng > 0 else 50
    zone = 'DISCOUNT 🟢 (BUY zone)' if current < eq else 'PREMIUM 🔴 (SELL zone)'

    print(f"📏 Dealing Range  (last {lookback} bars)")
    print(f"   Range High     : {rh:.2f}")
    print(f"   75% Level      : {l75:.2f}")
    print(f"   50% Equilibrium: {eq:.2f}  ← (Fair Value)")
    print(f"   25% Level      : {l25:.2f}")
    print(f"   Range Low      : {rl:.2f}")
    print(f"   ─────────────────────────")
    print(f"   Current Price  : {current:.2f}  ({pct:.1f}% in range)")
    print(f"   Zone           : {zone}")
    return {'zone': 'DISCOUNT' if current < eq else 'PREMIUM', 'eq': eq, 'l25': l25, 'l75': l75, 'rh': rh, 'rl': rl}

dealing = analyze_dealing_range(df)

📏 Dealing Range  (last 20 bars)
   Range High     : 67470.98
   75% Level      : 67374.68
   50% Equilibrium: 67278.39  ← (Fair Value)
   25% Level      : 67182.10
   Range Low      : 67085.80
   ─────────────────────────
   Current Price  : 67336.27  (65.0% in range)
   Zone           : PREMIUM 🔴 (SELL zone)


---
## 🏦 Concept 5 — Order Block (Institutional Footprint)
An **Order Block** is the last opposing candle before a significant move.  
- **Bullish OB**: Last bearish candle before a big bullish move
- **Bearish OB**: Last bullish candle before a big bearish move

Institutions leave unfilled orders here — price often returns to this zone.

In [7]:
# ═══════════════════════════════════════════════════
# CELL 7 — ORDER BLOCK DETECTION
# ═══════════════════════════════════════════════════
def detect_order_block(df, bias='BULLISH', lookback=15, min_move_pct=0.002):
    index  = len(df) - 1
    blocks = []

    for i in range(max(0, index-lookback), index-1):
        bar      = df.iloc[i]
        next_bar = df.iloc[i+1]
        change   = abs(next_bar['close'] - bar['close']) / bar['close']
        if bias == 'BULLISH' and change > min_move_pct and next_bar['close'] > bar['close']:
            blocks.append({'type': 'BULLISH', 'high': bar['high'], 'low': bar['low'], 'strength': min(change*10,1.0), 'bar_idx': i})
        elif bias == 'BEARISH' and change > min_move_pct and next_bar['close'] < bar['close']:
            blocks.append({'type': 'BEARISH', 'high': bar['high'], 'low': bar['low'], 'strength': min(change*10,1.0), 'bar_idx': i})

    if blocks:
        ob = blocks[-1]
        print(f"🏦 {ob['type']} Order Block Found")
        print(f"   OB High   : {ob['high']:.2f}")
        print(f"   OB Low    : {ob['low']:.2f}")
        print(f"   Strength  : {ob['strength']:.2%}")
        print(f"   Trade Idea: Enter when price returns to OB zone ({ob['low']:.2f} - {ob['high']:.2f})")
        return {**ob, 'detected': True}
    
    print(f"🏦 No {bias} Order Block found in last {lookback} bars")
    return {'detected': False}

bias_guess = 'BULLISH' if 'BULLISH' in mss.get('structure','') else 'BEARISH'
ob = detect_order_block(df, bias=bias_guess)

🏦 No BEARISH Order Block found in last 15 bars


---
## 🎯 Concept 6 — OTE (Optimal Trade Entry) — Fibonacci 62%-79%
ICT's **OTE Zone** = the 62% to 79% Fibonacci retracement of the FVG range.  
When price retraces INTO this zone after a sweep + FVG = **highest probability entry.**

In [8]:
# ═══════════════════════════════════════════════════
# CELL 8 — OTE (OPTIMAL TRADE ENTRY)
# ═══════════════════════════════════════════════════
def check_ote_levels(df, fvg):
    if not fvg.get('detected'):
        print("🎯 OTE: Requires an active FVG — run FVG cell first")
        return {'valid': False}
    current   = float(df['close'].iloc[-1])
    fvg_range = fvg['high'] - fvg['low']
    ote_62    = fvg['low'] + fvg_range * 0.62
    ote_70    = fvg['low'] + fvg_range * 0.70  # Sweet spot
    ote_79    = fvg['low'] + fvg_range * 0.79
    in_ote    = ote_62 <= current <= ote_79

    print(f"🎯 OTE Fibonacci Levels (from FVG {fvg['low']:.2f} - {fvg['high']:.2f})")
    print(f"   79% Level   : {ote_79:.2f}  ← OTE Zone Top")
    print(f"   70% Midpoint: {ote_70:.2f}  ← Sweet Spot")
    print(f"   62% Level   : {ote_62:.2f}  ← OTE Zone Bottom")
    print(f"   ─────────────────────────")
    print(f"   Current     : {current:.2f}")
    print(f"   In OTE Zone : {'✅ YES — Prime Entry!' if in_ote else '❌ Not yet in zone'}")
    return {'valid': in_ote, 'ote_62': ote_62, 'ote_70': ote_70, 'ote_79': ote_79}

ote = check_ote_levels(df, fvg)

🎯 OTE Fibonacci Levels (from FVG 67786.14 - 67851.40)
   79% Level   : 67837.70  ← OTE Zone Top
   70% Midpoint: 67831.82  ← Sweet Spot
   62% Level   : 67826.60  ← OTE Zone Bottom
   ─────────────────────────
   Current     : 67336.27
   In OTE Zone : ❌ Not yet in zone


---
## ⏰ Concept 7 — Silver Bullet Time Windows
ICT's **Silver Bullet** strategy only trades during 3 specific 1-hour windows (New York time):
- **3 AM – 4 AM** → London Open (most volatile)
- **10 AM – 11 AM** → New York AM Session Open
- **2 PM – 3 PM** → New York PM Session

Trading ONLY during these windows = higher win rate.

In [9]:
# ═══════════════════════════════════════════════════
# CELL 9 — SILVER BULLET TIME WINDOWS
# ═══════════════════════════════════════════════════
def check_silver_bullet_time(symbol=SYMBOL):
    tick = mt5.symbol_info_tick(symbol)
    server_time = datetime.fromtimestamp(tick.time) if tick else datetime.utcnow()
    offset = int(os.getenv("MT5_SERVER_TIME_OFFSET", 0))
    adj    = server_time + timedelta(hours=offset)
    h      = adj.hour

    windows = {3: "London Open", 10: "NY AM Session", 14: "NY PM Session"}
    is_sb   = h in windows
    name    = windows.get(h, "Outside Windows")

    all_windows = [(3,4,'London Open'), (10,11,'NY AM Session'), (14,15,'NY PM Session')]
    
    print(f"⏰ Silver Bullet Time Analysis")
    print(f"   Server Time  : {server_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Adjusted Hour: {h:02d}:00")
    print(f"   ─────────────────────────")
    for s, e, n in all_windows:
        active = '✅ ACTIVE NOW' if h == s else '   '
        print(f"   {active}  {s:02d}:00 - {e:02d}:00  →  {n}")
    print(f"   ─────────────────────────")
    print(f"   Status: {'🎯 PRIME ENTRY WINDOW — Trade aggresively!' if is_sb else '⏳ Wait for Silver Bullet window'}")
    return {'active': is_sb, 'hour': h, 'window': name}

sb = check_silver_bullet_time()

⏰ Silver Bullet Time Analysis
   Server Time  : 2026-03-08 04:19:49
   Adjusted Hour: 04:00
   ─────────────────────────
        03:00 - 04:00  →  London Open
        10:00 - 11:00  →  NY AM Session
        14:00 - 15:00  →  NY PM Session
   ─────────────────────────
   Status: ⏳ Wait for Silver Bullet window


---
## 🎯 Concept 8 — Liquidity Pool (Take Profit Target)
ICT says banks always target the **next pool of liquidity** — clusters of stop-loss orders sitting above swing highs or below swing lows.  
Use this as your **Take Profit target** — price is magnetically drawn there.

In [10]:
# ═══════════════════════════════════════════════════
# CELL 10 — LIQUIDITY POOL (TAKE PROFIT TARGETING)
# ═══════════════════════════════════════════════════
def find_liquidity_pool(df, direction='UP', lookback=30):
    index   = len(df) - 1
    current = float(df['close'].iloc[-1])
    recent  = df.iloc[max(0, index-lookback): index]

    if direction == 'UP':
        highs_above = recent['high'][recent['high'] > current]
        tp   = float(highs_above.min()) if not highs_above.empty else current * 1.02
        label = "Buy-Side Liquidity (above swing highs)"
    else:
        lows_below  = recent['low'][recent['low'] < current]
        tp   = float(lows_below.max()) if not lows_below.empty else current * 0.98
        label = "Sell-Side Liquidity (below swing lows)"

    dist = abs(tp - current)
    pct  = dist / current * 100
    print(f"🎯 Liquidity Pool ({direction})")
    print(f"   Type     : {label}")
    print(f"   Current  : {current:.2f}")
    print(f"   TP Target: {tp:.2f}  (Distance: {dist:.2f} = {pct:.2f}%)")
    return {'tp': tp, 'distance': dist, 'pct': pct}

direction = 'UP' if 'BULLISH' in mss.get('structure','') else 'DOWN'
lp = find_liquidity_pool(df, direction=direction)

🎯 Liquidity Pool (DOWN)
   Type     : Sell-Side Liquidity (below swing lows)
   Current  : 67336.27
   TP Target: 67324.42  (Distance: 11.85 = 0.02%)


---
## 🧠 Concept 9 — Full ICT Confluence Engine
Combines ALL concepts into a single signal. Requires **2.0+ score** to generate BUY/SELL.

| Signal | Score Weight |
|--------|-------------|
| Liquidity Sweep (aligned) | +1.0 |
| Fair Value Gap  (aligned) | +1.0 |
| Order Block     (aligned) | +1.0 |
| Dealing Range Zone        | +0.5 |
| OTE Zone active           | +0.5 |
| Silver Bullet Window      | +0.3 |

In [11]:
# ═══════════════════════════════════════════════════
# CELL 11 — FULL CONFLUENCE ENGINE
# ═══════════════════════════════════════════════════
def run_ict_confluence_engine(df):
    print("═" * 55)
    print("   🧠 ICT FULL CONFLUENCE ENGINE")
    print("═" * 55)
    df2 = df.copy()
    # Add indicators
    df2['sma20'] = df2['close'].rolling(20).mean()
    delta = df2['close'].diff()
    gain  = (delta.where(delta>0,0)).rolling(14).mean()
    loss  = (-delta.where(delta<0,0)).rolling(14).mean()
    df2['rsi'] = 100 - (100/(1+gain/loss))

    mss_r     = detect_market_structure(df2)
    bias      = 'BULLISH' if 'BULLISH' in mss_r['structure'] else ('BEARISH' if 'BEARISH' in mss_r['structure'] else 'NEUTRAL')

    if bias == 'NEUTRAL':
        print("\n⚪ SIGNAL: HOLD — No clear market structure")
        return {'action': 'HOLD', 'bias': 'NEUTRAL', 'confidence': 0.0, 'score': 0.0}

    sweep_r   = detect_liquidity_sweep(df2)
    fvg_r     = detect_fair_value_gap(df2)
    dealing_r = analyze_dealing_range(df2)
    ob_r      = detect_order_block(df2, bias=bias)
    ote_r     = check_ote_levels(df2, fvg_r)
    sb_r      = check_silver_bullet_time()
    lp_r      = find_liquidity_pool(df2, direction='UP' if bias=='BULLISH' else 'DOWN')

    # Scoring
    score, signals, strengths = 0.0, [], []
    if sweep_r['detected']:
        if (bias=='BULLISH' and sweep_r['type']=='BELOW_LOW') or (bias=='BEARISH' and sweep_r['type']=='ABOVE_HIGH'):
            score+=1.0; signals.append("✅ Liquidity Sweep"); strengths.append(sweep_r['strength'])
    if fvg_r.get('detected') and fvg_r.get('type')==bias:
        score+=1.0; signals.append("✅ FVG"); strengths.append(fvg_r['strength'])
    if ob_r.get('detected') and ob_r.get('type')==bias:
        score+=1.0; signals.append("✅ Order Block"); strengths.append(ob_r['strength'])
    if dealing_r['zone']==('DISCOUNT' if bias=='BULLISH' else 'PREMIUM'):
        score+=0.5; signals.append("✅ Dealing Zone")
    if ote_r.get('valid'):
        score+=0.5; signals.append("✅ OTE (62-79%)")
    if sb_r['active']:
        score+=0.3; signals.append(f"✅ Silver Bullet ({sb_r['window']})")

    avg_str    = sum(strengths)/len(strengths) if strengths else 0
    confidence = min(avg_str + min(len(strengths)*0.1, 0.3), 1.0)
    action     = ('BUY' if bias=='BULLISH' else 'SELL') if score >= 2.0 else 'HOLD'
    current    = float(df2['close'].iloc[-1])

    print(f"\n{'─'*55}")
    print(f"  Bias       : {bias}")
    print(f"  Score      : {score:.1f} / 4.3")
    print(f"  Confidence : {confidence:.0%}")
    print(f"  Signals    :")
    for s in signals: print(f"               {s}")
    print(f"{'─'*55}")
    if action != 'HOLD':
        sl = sweep_r.get('swept_level', current*(0.98 if action=='BUY' else 1.02))
        print(f"  🏹 SIGNAL  : {action}")
        print(f"  💰 Entry   : {current:.2f}")
        print(f"  🛑 SL      : {sl:.2f}")
        print(f"  🎯 TP      : {lp_r['tp']:.2f}")
    else:
        print(f"  ⏳ SIGNAL  : HOLD (need 2.0+, got {score:.1f})")
    print("═" * 55)
    return {'action': action, 'bias': bias, 'confidence': confidence, 'score': score, 'signals': signals}

signal = run_ict_confluence_engine(df)

═══════════════════════════════════════════════════════
   🧠 ICT FULL CONFLUENCE ENGINE
═══════════════════════════════════════════════════════
⚪ Market Structure : NEUTRAL
   Prev High : 67375.64  |  Curr High : 67375.32
   Prev Low  : 67288.39  |  Curr Low  : 67308.49

⚪ SIGNAL: HOLD — No clear market structure


---
## 📈 Concept 10 — Walk-Forward Backtest
Tests the ICT confluence engine on historical bars.  
For each signal, checks price after **5 bars** to determine win/loss.

In [12]:
# ═══════════════════════════════════════════════════
# CELL 12 — WALK-FORWARD BACKTEST
# ═══════════════════════════════════════════════════
def backtest_ict(df, start_bar=100, min_score=1.5, forward_bars=5):
    print(f"📈 Walk-Forward Backtest | Min Score: {min_score} | Forward: {forward_bars} bars")
    print("─" * 55)
    results = []
    for i in range(start_bar, len(df) - forward_bars):
        window = df.iloc[:i+1].copy()
        try:
            mss_r   = detect_market_structure(window)
            bias    = 'BULLISH' if 'BULLISH' in mss_r['structure'] else ('BEARISH' if 'BEARISH' in mss_r['structure'] else 'NEUTRAL')
            if bias == 'NEUTRAL': continue
            sweep_r = detect_liquidity_sweep(window)
            fvg_r   = detect_fair_value_gap(window)
            score   = 0
            if sweep_r['detected']:
                if (bias=='BULLISH' and sweep_r['type']=='BELOW_LOW') or (bias=='BEARISH' and sweep_r['type']=='ABOVE_HIGH'):
                    score += 1
            if fvg_r.get('detected') and fvg_r.get('type') == bias:
                score += 1
            if score >= min_score:
                entry  = float(window['close'].iloc[-1])
                future = df['close'].iloc[i+1:i+1+forward_bars]
                action = 'BUY' if bias == 'BULLISH' else 'SELL'
                exit_p = float(future.iloc[-1])
                pnl    = (exit_p - entry) if action == 'BUY' else (entry - exit_p)
                results.append({'bar': i, 'action': action, 'entry': entry, 'exit': exit_p, 'pnl': pnl, 'win': pnl > 0})
        except: continue

    if not results:
        print("❌ No signals generated")
        return []

    total     = len(results)
    wins      = sum(1 for r in results if r['win'])
    total_pnl = sum(r['pnl'] for r in results)
    win_rate  = wins / total * 100
    avg_win   = sum(r['pnl'] for r in results if r['win']) / max(wins, 1)
    avg_loss  = sum(r['pnl'] for r in results if not r['win']) / max(total-wins, 1)

    print(f"  Total Signals : {total}")
    print(f"  Wins          : {wins}  ({win_rate:.1f}%)")
    print(f"  Losses        : {total - wins}")
    print(f"  Total PnL     : {total_pnl:+.2f} pts")
    print(f"  Avg Win       : {avg_win:+.2f} pts")
    print(f"  Avg Loss      : {avg_loss:+.2f} pts")
    if avg_loss != 0:
        rr = abs(avg_win / avg_loss)
        print(f"  R:R Ratio     : 1:{rr:.2f}")
    print("─" * 55)
    print("  Last 5 Trades:")
    for r in results[-5:]:
        e = '✅' if r['win'] else '❌'
        print(f"  {e} {r['action']:4s} {r['entry']:.2f} → {r['exit']:.2f}  ({r['pnl']:+.2f} pts)")
    return results

results = backtest_ict(df, start_bar=100, min_score=1.5, forward_bars=5)

📈 Walk-Forward Backtest | Min Score: 1.5 | Forward: 5 bars
───────────────────────────────────────────────────────
⚪ Market Structure : NEUTRAL
   Prev High : 69996.14  |  Curr High : 69276.48
   Prev Low  : 68872.36  |  Curr Low  : 68872.36
⚪ Market Structure : NEUTRAL
   Prev High : 69962.90  |  Curr High : 69189.90
   Prev Low  : 68638.67  |  Curr Low  : 68638.67
⚪ Market Structure : NEUTRAL
   Prev High : 69276.48  |  Curr High : 68884.36
   Prev Low  : 68634.39  |  Curr Low  : 68634.39
⚪ Market Structure : NEUTRAL
   Prev High : 69189.90  |  Curr High : 68994.64
   Prev Low  : 68634.39  |  Curr Low  : 68717.56
⚪ Market Structure : NEUTRAL
   Prev High : 69027.43  |  Curr High : 69027.43
   Prev Low  : 68634.39  |  Curr Low  : 68882.83
⚪ Market Structure : NEUTRAL
   Prev High : 69096.64  |  Curr High : 69096.64
   Prev Low  : 68717.56  |  Curr Low  : 68878.33
⚪ Market Structure : NEUTRAL
   Prev High : 69096.64  |  Curr High : 69094.15
   Prev Low  : 68727.81  |  Curr Low  : 68727

---
## ✅ Notebook Complete

### Summary
| Concept | Purpose |
|---------|--------|
| MSS/BOS/ChoCH | Determine trend direction |
| Liquidity Sweep | Detect stop hunts for reversal |
| FVG | Find imbalance zones for entry |
| Dealing Range | Confirm buy/sell zone alignment |
| Order Block | Find institutional entry zones |
| OTE | Fibonacci precision entry |
| Silver Bullet | Time-based probability boost |
| Liquidity Pool | Set accurate take profit |

> **Next Step:** Modify parameters in each cell and observe how signals change. Try different symbols and timeframes!

Built by **Next Level Brain** 🧠